# 51 — Ground Station Provenance Validation
Detect station aliasing and distance-risk across sites.

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def normalise_date_col(df):
    for c in df.columns:
        if c.lower() in {"date","day","datetime","timestamp","time","time_utc","publishedat","published_at"}:
            out = df.copy()
            out["date"] = pd.to_datetime(out[c], errors="coerce").dt.normalize()
            return out, c
    return df.copy(), None

def list_output_files():
    rows = []
    if not OUTPUTS.exists():
        return pd.DataFrame(columns=["path","size_bytes","sha256"])
    for p in sorted(OUTPUTS.rglob("*")):
        if p.is_file():
            rows.append({
                "path": str(p.relative_to(PROJECT_ROOT)),
                "size_bytes": p.stat().st_size,
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

run_cfg = load_yaml(CONFIGS / "run.yml")
phase_cfg = load_yaml(CONFIGS / "phase50.yml")
sites_df = pd.DataFrame(run_cfg.get("sites", []))
phase_sites = pd.DataFrame(phase_cfg.get("additional_sites", []))
if not phase_sites.empty:
    existing = set(sites_df.get("id", pd.Series(dtype=str)).astype(str).tolist()) if not sites_df.empty else set()
    phase_sites = phase_sites[~phase_sites["id"].astype(str).isin(existing)]
    sites_df = pd.concat([sites_df, phase_sites], ignore_index=True)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])
date_from = run_cfg.get("run", {}).get("date_from", "2025-01-01")
date_to = run_cfg.get("run", {}).get("date_to", "2025-01-31")

In [ ]:
step = "51_station_provenance_validation"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

station_cfg = load_yaml(CONFIGS / "station_overrides.yml")
max_dist_km = float(station_cfg.get("validation", {}).get("max_station_distance_km", 25))
alias_allowed = bool(station_cfg.get("validation", {}).get("allow_shared_station_ids", False))

def hav_km(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan
    r = 6371.0
    p1, p2 = np.radians([lat1, lat2])
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2 * r * np.arctan2(np.sqrt(a), np.sqrt(1-a))

records = []
for _, site in sites_df.iterrows():
    sid = str(site.get("id", ""))
    raw_path = OUTPUTS / "04_ground_aq_providers" / f"waqi_{sid}_raw.json"
    if raw_path.exists():
        raw = load_json(raw_path)
        data = raw.get("data", {})
        city = data.get("city", {}) if isinstance(data, dict) else {}
        station_name = city.get("name") if isinstance(city, dict) else None
        geo = city.get("geo") if isinstance(city, dict) else None
        station_lat = geo[0] if isinstance(geo, list) and len(geo) >= 2 else np.nan
        station_lon = geo[1] if isinstance(geo, list) and len(geo) >= 2 else np.nan
        station_id = station_name or data.get("idx") or "unknown"
        dist = hav_km(site.get("lat"), site.get("lon"), station_lat, station_lon)
    else:
        station_id, station_name, station_lat, station_lon, dist = "", "", np.nan, np.nan, np.nan
    records.append({
        "site_id": site.get("id"),
        "site_name": site.get("name"),
        "role": site.get("role"),
        "provider_station_id": str(station_id),
        "provider_station_name": station_name,
        "provider_station_lat": station_lat,
        "provider_station_lon": station_lon,
        "station_distance_km": dist,
        "raw_path": str(raw_path.relative_to(PROJECT_ROOT)) if raw_path.exists() else "",
    })

station_df = pd.DataFrame(records)
if not station_df.empty:
    station_df["distance_flag"] = np.where(station_df["station_distance_km"].fillna(max_dist_km + 1) > max_dist_km, "review", "ok")
    alias_counts = station_df.groupby("provider_station_id").size().rename("site_count").reset_index()
    station_df = station_df.merge(alias_counts, on="provider_station_id", how="left")
    station_df["alias_flag"] = np.where((station_df["site_count"].fillna(0) > 1) & (~alias_allowed) & (station_df["provider_station_id"] != ""), "review", "ok")

station_df.to_csv(out / "station_provenance.csv", index=False)
summary = {
    "sites_with_station_match": int((station_df["provider_station_id"].astype(str) != "").sum()) if not station_df.empty else 0,
    "sites_flagged_distance": int((station_df["distance_flag"] == "review").sum()) if not station_df.empty else 0,
    "sites_flagged_aliasing": int((station_df["alias_flag"] == "review").sum()) if not station_df.empty else 0,
}
write_json(out / "station_validation_summary.json", summary)

manifest = manifest_base(step, [CONFIGS / "run.yml", CONFIGS / "station_overrides.yml"])
for p in [out / "station_provenance.csv", out / "station_validation_summary.json"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
manifest["summary"] = summary
write_json(out / "manifest.json", manifest)
display(station_df)
print(summary)